In [37]:
import json
from typing import List, Dict

class DatasetLoader:
    @staticmethod
    def load_aggregated_dataset(file_path: str, verbose: bool = False) -> List[Dict]:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        raw_dataset = data["entries"]

        dataset = []
        for entry in raw_dataset:
            dataset_name = entry.get("dataset_name")
            annotations = entry.get("annotations", {})

            # Determina il label in base al dataset
            if dataset_name == "MultiPICo":
                label = annotations.get("aggregate_irony_label")
            else:  # dataset CSC
                label = annotations.get("aggregate_sarcasm_label")

            # Converti context_type in 1/0/null
            context_type_raw = entry.get("context_type")
            if context_type_raw is None:
                context_type = None
            else:
                context_type = 1 if context_type_raw == "non_neutral" else 0

            cleaned_entry = {
                "unique_id": entry.get("unique_id"),
                "source": entry.get("source"),
                "context_type": context_type,
                "language": entry.get("language"),
                "context": entry.get("context"),
                "post": entry.get("post"),
                "reply": entry.get("reply"),
                "dialogue": entry.get("dialogue"),
                "label": label,
            }

            dataset.append(cleaned_entry)
        
        return dataset

dataset_name = 'MultiPICo' #CSC
file_path = f"output/final_dataset/{dataset_name}_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered.json"

dataset = DatasetLoader.load_aggregated_dataset(file_path)
dataset[0]

{'unique_id': '8130940562900296_16285417133666056067',
 'source': 'twitter',
 'context_type': None,
 'language': 'en',
 'context': None,
 'post': 'No one is curious as to why the war in Ukraine has almost zero footage?\n\nNo one?',
 'reply': 'Take a look at Operator Starsky on YouTube.',
 'dialogue': {'turns': [{'u1': "I'm starting to wonder if I'm the only one who's noticed this, have we all become numb to the global news?",
    'u2': 'Nah, I think people are just distracted by more current stuff, or is it just me?'},
   {'u1': "I don't get it, I mean, we see live streams of concerts and stuff, but somehow the war in Ukraine is still under the radar?",
    'u2': "I think it's a mix of both, and also the fact that most people aren't really invested in international news"},
   {'u1': 'No one is curious as to why the war in Ukraine has almost zero footage? No one?',
    'u2': 'Take a look at Operator Starsky on YouTube.'}]},
 'label': 0}

In [38]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

def intra_turn_similarity(turns, method="bow"):
    texts1 = [t["u1"] for t in turns]
    texts2 = [t["u2"] for t in turns]

    all_texts = texts1 + texts2

    if method == "bow":
        vectorizer = CountVectorizer()
        X = vectorizer.fit_transform(all_texts)
    elif method == "tfidf":
        vectorizer = TfidfVectorizer()
        X = vectorizer.fit_transform(all_texts)
    elif method == "st":
        embeddings = model.encode(all_texts, normalize_embeddings=True)
        X = embeddings
    else:
        raise ValueError("Unknown method")

    sim_matrix = cosine_similarity(X)

    scores = []
    n = len(turns)
    for i in range(n):
        scores.append(sim_matrix[i, i + n])  # u1 vs u2

    return scores

def extra_turn_similarity(turn_strings, method="bow"):
    if len(turn_strings) < 2:
        return []

    if method == "bow":
        vectorizer = CountVectorizer()
        X = vectorizer.fit_transform(turn_strings)
    elif method == "tfidf":
        vectorizer = TfidfVectorizer()
        X = vectorizer.fit_transform(turn_strings)
    elif method == "st":
        X = model.encode(turn_strings, normalize_embeddings=True)
    else:
        raise ValueError("Unknown method")

    sim_matrix = cosine_similarity(X)

    scores = []
    pairs = []

    for i in range(len(turn_strings)):
        for j in range(i + 1, len(turn_strings)):
            scores.append(sim_matrix[i, j])
            pairs.append((i, j))

    return scores, pairs

def starts_with_first_message(turns):
    for t in turns:
        if t["u1"].strip().startswith("First message") or t["u2"].strip().startswith("First message"):
            return True
    return False

In [39]:
import tqdm

def extract_turns(dialogue):
    turn_strings = []
    structured_turns = []
    total_turns = []

    turn_number = 0
    for turn in dialogue["turns"]:
        turn_number += 1

        u1 = turn["u1"]
        u2 = turn["u2"]

        turn_strings.append(f"{u1} {u2}")
        total_turns.append({"u1": u1, "u2": u2})

        if not (dataset_name == "MultiPICo" and turn_number == 3):
            structured_turns.append({"u1": u1, "u2": u2})

    return total_turns, structured_turns, turn_strings


THRESHOLD = 0.65
OUTPUT_FILE = "high_similarity_dialogues.txt"

count = 0

entries_to_be_removed = []
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    for entry in tqdm.tqdm(dataset, desc="Processing entries"):
        total_turns, structured_turns, turn_strings = extract_turns(entry["dialogue"])

        # 🔴 NUOVO CONTROLLO
        has_first_message = starts_with_first_message(total_turns)

        intra_scores = intra_turn_similarity(structured_turns, method="bow")
        extra_scores, pairs = extra_turn_similarity(turn_strings, method="bow")

        high_intra = [s for s in intra_scores if s > THRESHOLD]
        high_extra = [(s, p) for s, p in zip(extra_scores, pairs) if s > THRESHOLD]

        # 🔴 AGGIUNGI condizione
        if high_intra or high_extra or has_first_message:
            entries_to_be_removed.append(entry["unique_id"])
            count += 1

            def write(line=""):
                print(line)
                f.write(line + "\n")

            write(f"\nEntry ID: {entry['unique_id']}")

            if has_first_message:
                write("⚠️ Contains message starting with 'First message'")

            write("Dialogue:")
            for i, turn in enumerate(total_turns):
                write(f"Turn {i}")
                write(f"u1: {turn['u1']}")
                write(f"u2: {turn['u2']}")
                write()

            write("=== HIGH INTRA-TURN SIMILARITIES ===")
            for i, s in enumerate(intra_scores):
                if s > THRESHOLD:
                    write(f"Turn {i}: {s:.3f}")

            write("=== HIGH EXTRA-TURN SIMILARITIES ===")
            for (score, (i, j)) in high_extra:
                write(f"Turn {i} - Turn {j}: {score:.3f}")

            write("-" * 50)

    final_msg = f"\nTotal entries with high similarity: {count} out of {len(dataset)}"
    f.write(final_msg + "\n")


Processing entries:  10%|█         | 145/1382 [00:00<00:01, 720.84it/s]


Entry ID: 666389135429989758_11849521393128529416
Dialogue:
Turn 0
u1: Your parents are more important to who you become than your own choices. I just don't believe you can change who you are with a simple label or a fancy title.
u2: But that's not true! You can totally change who you are with the right attitude and hard work. It's not that simple to change, but you can still grow and develop!

Turn 1
u1: You can try to change who you are, but it's not going to make you truly happy. You're still going to be the same person underneath all the pretenses and labels.
u2: How can you say that? It's because you've never tried to change for yourself. You've never been in a position where you've had to face your fears and doubts and really transform who you are.

Turn 2
u1: No, you are what you are born as. NO IDENTIFYING
u2: Jeepers, no need to shout!

=== HIGH INTRA-TURN SIMILARITIES ===
=== HIGH EXTRA-TURN SIMILARITIES ===
Turn 0 - Turn 1: 0.685
--------------------------------------------

Processing entries:  32%|███▏      | 436/1382 [00:00<00:01, 716.24it/s]


Entry ID: 4781505017412451826_9647404868022358629
Dialogue:
Turn 0
u1: u1: u1: Hmm, the way I solved numbers 1-3 was by just going up the pyramid and counting the dots. That worked for me so far.
u2: u2: u2: Yeah, that's what I tried first, but it's a lot harder than it seems. You have to use the digits to unlock the pattern.

Turn 1
u1: u1: It sounds like there's more to this than just counting dots? What do you mean by "unlock the pattern"?
u2: u2: I mean that you have to use the digits to get to the next level. It's not just a matter of going up the pyramid.

Turn 2
u1: 10/12 number 4 & 8 have me stumped
u2: I just got 8, it's easier if you try it in reverse order.

=== HIGH INTRA-TURN SIMILARITIES ===
=== HIGH EXTRA-TURN SIMILARITIES ===
Turn 0 - Turn 1: 0.714
--------------------------------------------------


Processing entries:  53%|█████▎    | 727/1382 [00:01<00:00, 715.75it/s]

...

Processing entries:  63%|██████▎   | 871/1382 [00:01<00:00, 698.35it/s]


Entry ID: 10449167530093250686_17703860232231147136
Dialogue:
Turn 0
u1: Just asked about the display quality of the new Galaxy Z Flip3. Anybody else having the same concern? #FlipAtFlipkart #GalaxyZFlip3 #stufflistingsarmy
u2: Honestly, I don't think display quality is a major concern. The Galaxy Z Flip3 has gotten great reviews so far #FlipAtFlipkart #GalaxyZFlip3 #stufflistingsarmy

Turn 1
u1: Okay, but what about the display type? Dynamic AMOLED or something else? Anyone know? #FlipAtFlipkart #GalaxyZFlip3 #stufflistingsarmy
u2: I think it's Dynamic AMOLED 2X. Someone should check the official specs #FlipAtFlipkart #GalaxyZFlip3 #stufflistingsarmy

Turn 2
u1: Q5. What is the display type of the #GalaxyZFlip3? A. Dynamic AMOLED 2X B. Dynamic AMOLED 3X C. Static AMOLED X D. Dynamic IPS #FlipAtFlipkart #GalaxyZFlip3 #GalaxyZFlip3Giveaway #stufflistingsarmy
u2: A. Dynamic AMOLED 2X #FlipAtFlipkart #GalaxyZFlip3 #GalaxyZFlip3Giveaway #stufflistingsarmy

=== HIGH INTRA-TURN SIMILARITIES

Processing entries:  78%|███████▊  | 1084/1382 [00:01<00:00, 702.29it/s]


Entry ID: 12781958716603294319_5422444308977562625
Dialogue:
Turn 0
u1: It's clear to me that the current government is in the pockets of big business, they're only looking out for their corporate mates, not the people. We need a change, but I'm not sure Labor is the answer.
u2: I completely agree with you! Labor seems to be stuck in the same old ways of thinking, they're not willing to take a stand against the corporations that really run this country. We need a party that's going to challenge the status quo and fight for real change.

Turn 1
u1: But how do we bring about real change? We need a party that's got the guts to take on the powerful and the corrupt. And I just don't see that in Labor right now.
u2: That's exactly why I think the Greens are the way to go. They're not afraid to take on the big boys and they've got a real commitment to creating a more equal society. Plus, they're not beholden to the same corporate interests as the other parties.

Turn 2
u1: Labor was in gover

Processing entries: 100%|██████████| 1382/1382 [00:01<00:00, 705.80it/s]


In [40]:

# Remove entries_to_be_removed from the original filtered file and save as final

entries_to_remove_set = set(entries_to_be_removed)

with open(file_path, 'r', encoding='utf-8') as f:
    original_data = json.load(f)

original_entries = original_data["entries"]
filtered_entries = [e for e in original_entries if e.get("unique_id") not in entries_to_remove_set]

print(f"Original entries: {len(original_entries)}")
print(f"Removed entries:  {len(original_entries) - len(filtered_entries)}")
print(f"Final entries:    {len(filtered_entries)}")

output_data = {**original_data, "entries": filtered_entries}

output_file_path = f"output/final_dataset/{dataset_name}_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_final.json"
with open(output_file_path, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved to: {output_file_path}")


Original entries: 1382
Removed entries:  5
Final entries:    1377

Saved to: output/final_dataset/MultiPICo_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_final.json

Saved to: output/final_dataset/MultiPICo_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_final.json
